# Chapter 6, Guide 4: Quantum search as Hamiltonian simulation

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 6,
"Building the simulation-based search".

The Hamiltonian is $H = |w\rangle\langle w| + |s\rangle\langle s|$. Evolving $|s\rangle$ under
it for $t^* = \tfrac{\pi}{2}\sqrt N$ lands on $|w\rangle$. The circuit Trotterizes
$e^{-iHt}$ into $K$ steps of $e^{-i|w\rangle\langle w|\Delta t}\,e^{-i|s\rangle\langle s|\Delta t}$;
each step calls the oracle twice (compute, then uncompute). The $\Delta t=\pi$ shortcut
turns the whole thing into Grover's algorithm.

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator

bq = lambda j, tot: tot - 1 - j

## Test 1: dense evolution gives $\Pr(w)=1$ at $t^*=\tfrac{\pi}{2}\sqrt N$

In [2]:
n, w = 3, 5
N = 2**n
e_w = np.zeros(N); e_w[w] = 1
s = np.ones(N) / np.sqrt(N)
H = np.outer(e_w, e_w) + np.outer(s, s)

t_star = np.pi / 2 * np.sqrt(N)
psi = expm(-1j * H * t_star) @ s
print(f"t* = {t_star:.4f},  Pr(w) = {abs(psi[w])**2:.10f}")
assert abs(abs(psi[w]) ** 2 - 1) < 1e-10
ts = np.linspace(0, 2 * t_star, 200)
plt.plot(ts, [abs((expm(-1j * H * t) @ s)[w]) ** 2 for t in ts])
plt.axvline(t_star, ls="--", c="k"); plt.xlabel("t"); plt.ylabel("Pr(w)"); plt.show()

t* = 4.4429,  Pr(w) = 1.0000000000


## The two exponentials as circuits

$e^{-i|w\rangle\langle w|\Delta t}$ puts the phase $e^{-i\Delta t}$ on $|w\rangle$ alone. With the
bit-flip oracle onto a clean ancilla: compute $f$ into the ancilla, apply the phase
gate $P(-\Delta t)$ to the ancilla, uncompute. Two oracle calls per step, as the book
counts.

$e^{-i|s\rangle\langle s|\Delta t} = H^{\otimes n} e^{-i|0\rangle\langle 0|\Delta t} H^{\otimes n}$, and the
phase on $|0\cdots0\rangle$ is an $X$-sandwiched multi-controlled phase gate.

In [3]:
def oracle_flip(qc, search, anc, w, n):
    bits = [(w >> (n - 1 - j)) & 1 for j in range(n)]
    zeros = [search[j] for j, b in enumerate(bits) if b == 0]
    if zeros: qc.x(zeros)
    qc.mcx(search, anc)
    if zeros: qc.x(zeros)


def exp_w(qc, search, anc, w, n, dt):
    oracle_flip(qc, search, anc, w, n)
    qc.p(-dt, anc)
    oracle_flip(qc, search, anc, w, n)


def exp_s(qc, search, n, dt):
    qc.h(search); qc.x(search)
    qc.mcp(-dt, search[:-1], search[-1])
    qc.x(search); qc.h(search)


def trotter_search(n, w, dt, K, symmetric=False):
    tot = n + 1
    search = [bq(j, tot) for j in range(n)]
    anc = bq(n, tot)
    qc = QuantumCircuit(tot)
    qc.h(search)
    for _ in range(K):
        if symmetric:
            exp_s(qc, search, n, dt / 2); exp_w(qc, search, anc, w, n, dt); exp_s(qc, search, n, dt / 2)
        else:
            exp_w(qc, search, anc, w, n, dt); exp_s(qc, search, n, dt)
    return qc


def search_register_unitary(n, w, dt, K, symmetric=False):
    """Unitary on the search register (ancilla stays |0>), without the initial H layer."""
    tot = n + 1
    search = [bq(j, tot) for j in range(n)]
    anc = bq(n, tot)
    qc = QuantumCircuit(tot)
    for _ in range(K):
        if symmetric:
            exp_s(qc, search, n, dt / 2); exp_w(qc, search, anc, w, n, dt); exp_s(qc, search, n, dt / 2)
        else:
            exp_w(qc, search, anc, w, n, dt); exp_s(qc, search, n, dt)
    U = Operator(qc).data
    return U[0::2, 0::2]                       # ancilla (Qiskit qubit 0) = 0 block

## Test 2: the split pieces are exact, and the error scales as the book says

Each factor alone is exact. The error of the product comes from the non-commuting
split. For a *single* step the first-order error falls as $\Delta t^2$; over the fixed
total time $t^*$ it accumulates to $O(t^*\Delta t)$ for the first-order split and
$O(\Delta t^2)$ for the symmetric split (Chapter 7 derives both slopes). A slope lower
than expected means a bug in the splitting order.

In [4]:
Uw_exact = lambda dt: expm(-1j * np.outer(e_w, e_w) * dt)
Us_exact = lambda dt: expm(-1j * np.outer(s, s) * dt)
tot = n + 1; search = [bq(j, tot) for j in range(n)]; anc = bq(n, tot)
qc = QuantumCircuit(tot); exp_w(qc, search, anc, w, n, 0.7)
assert np.allclose(Operator(qc).data[0::2, 0::2], Uw_exact(0.7))
qc = QuantumCircuit(tot); exp_s(qc, search, n, 0.7)
assert np.allclose(Operator(qc).data[0::2, 0::2], Us_exact(0.7))
print("each factor is exact")

dts = np.array([0.8, 0.4, 0.2, 0.1, 0.05])
one_step = [np.linalg.norm(search_register_unitary(n, w, dt, 1) - expm(-1j * H * dt), 2) for dt in dts]
print("single-step error slope (first order):",
      np.round(np.polyfit(np.log(dts), np.log(one_step), 1)[0], 2), " (expect 2)")

Ks = np.array([8, 16, 32, 64, 128])
dts_K = t_star / Ks
U_exact = expm(-1j * H * t_star)
err1 = [np.linalg.norm(search_register_unitary(n, w, dt, K) - U_exact, 2) for dt, K in zip(dts_K, Ks)]
err2 = [np.linalg.norm(search_register_unitary(n, w, dt, K, symmetric=True) - U_exact, 2) for dt, K in zip(dts_K, Ks)]
print("accumulated error slope at fixed t*, first order:", np.round(np.polyfit(np.log(dts_K), np.log(err1), 1)[0], 2), "(expect 1)")
print("accumulated error slope at fixed t*, symmetric  :", np.round(np.polyfit(np.log(dts_K), np.log(err2), 1)[0], 2), "(expect 2)")
plt.loglog(dts_K, err1, "o-", label="first order"); plt.loglog(dts_K, err2, "s-", label="symmetric")
plt.xlabel(r"$\Delta t$"); plt.ylabel(r"$\|U_{\rm circuit}-e^{-iHt^*}\|$"); plt.legend(); plt.show()

each factor is exact
single-step error slope (first order): 1.99  (expect 2)


accumulated error slope at fixed t*, first order: 1.0 (expect 1)
accumulated error slope at fixed t*, symmetric  : 2.0 (expect 2)


## Test 3: with $K\Delta t = t^*$ the circuit finds $w$

In [5]:
K = 128
sv = Statevector(trotter_search(n, w, t_star / K, K, symmetric=True))
print("Pr(w) with K=128 symmetric steps:", round(sv.probabilities(search[::-1])[w], 6))

Pr(w) with K=128 symmetric steps: 1.0


## The $\Delta t=\pi$ shortcut is Grover

At $\Delta t=\pi$ the two factors become $I-2|w\rangle\langle w|$ (the phase oracle) and
$I-2|s\rangle\langle s|$ (minus the diffuser), and $\pi/\varphi \approx (\pi/4)\sqrt N$ steps suffice.

In [6]:
G_from_split = search_register_unitary(n, w, np.pi, 1)
Uf = np.eye(N) - 2 * np.outer(e_w, e_w)
V = 2 * np.outer(s, s) - np.eye(N)
assert np.allclose(G_from_split, -V @ Uf)
print("one dt=pi step = -(diffuser)(oracle) = Grover iteration up to sign")
r = int(np.floor(np.pi / 4 * np.sqrt(N)))
sv = Statevector(trotter_search(n, w, np.pi, r))
print(f"Pr(w) after r={r} steps of dt=pi: {sv.probabilities(search[::-1])[w]:.4f}  (Grover value sin^2((2r+1)theta) = {np.sin((2*r+1)*np.arcsin(1/np.sqrt(N)))**2:.4f})")
print(f"oracle calls: Grover-style {r}, versus {2*K} for the K={K} Trotter run above")

one dt=pi step = -(diffuser)(oracle) = Grover iteration up to sign
Pr(w) after r=2 steps of dt=pi: 0.9453  (Grover value sin^2((2r+1)theta) = 0.9453)
oracle calls: Grover-style 2, versus 256 for the K=128 Trotter run above
